In [1]:
import numpy as np
import pandas as pd

In [2]:
def compute_rmse(pred, realized):
    T = len(pred)
    frob_sq = np.array([np.sum((pred[t] - realized[t])**2) for t in range(T)])
    return np.sqrt(frob_sq.mean())

def compute_qlike(pred, realized):
    T = len(pred)
    n = pred.shape[1]
    qlikes = []
    skipped = 0
    for t in range(T):
        try:
            Hhat_inv = np.linalg.inv(pred[t])
            A = realized[t] @ Hhat_inv
            sign, logdet = np.linalg.slogdet(A)
            if sign <= 0:
                skipped += 1
                continue
            qlikes.append(np.trace(A) - logdet - n)
        except np.linalg.LinAlgError:
            skipped += 1
            continue
    if skipped > 0:
        print(f"  Warning: skipped {skipped}/{T} time steps")
    return np.mean(qlikes)

def vech_to_sym(row):
    M = np.zeros((n, n))
    M[tril_idx] = row
    return M + M.T - np.diag(np.diag(M))

In [7]:
n = 8
tril_idx = np.tril_indices(n)

def vech_to_sym(row):
    M = np.zeros((n, n))
    M[tril_idx] = row
    return M + M.T - np.diag(np.diag(M))

def load_rc(filename):
    df = pd.read_csv(filename, sep='\t', header=None)
    df.columns = range(df.shape[1])
    return np.stack([vech_to_sym(df.iloc[t].values) for t in range(len(df))])

realized_rc = load_rc("data/rc_kernel.txt")
pred_chol   = load_rc("data/pred_chol_sub.txt")
pred_log    = load_rc("data/pred_log_sub.txt")
pred_dcc    = load_rc("data/pred_dcc_sub.txt")
pred_dcc_re = load_rc("data/pred_dcc_real_sub.txt")
pred_lstm   = load_rc("data/pred_lstm_rc_sub.txt")
pred_gru   = load_rc("data/pred_gru_rc_sub.txt")
pred_lstm_l   = load_rc("data/pred_lstm_logexp_rc_sub.txt")
pred_gru_l   = load_rc("data/pred_gru_logexp_rc_sub.txt")
pred_har_chol_ker = load_rc("data/pred_chol_kernel.txt")
pred_har_log_ker = load_rc("data/pred_log_kernel.txt")
pred_dcc_re_ker = load_rc("data/pred_dcc_real_kernel.txt")
pred_lstm_ker   = load_rc("data/pred_lstm_rc_kernel.txt")
pred_gru_ker   = load_rc("data/pred_gru_rc_kernel.txt")
pred_lstm_l_ker   = load_rc("data/pred_lstm_logexp_rc_kernel.txt")
pred_gru_l_ker   = load_rc("data/pred_gru_logexp_rc_kernel.txt")

n_test = len(pred_chol)
realized_rc = realized_rc[-n_test:]

predictions = {
    "HAR-Cholesky": pred_chol,
    "HAR-LogExp":   pred_log,
    "DCC-st": pred_dcc,
    "DCC-real": pred_dcc_re,
    "LSTM-Cholesky": pred_lstm,
    "GRU-Cholesky": pred_gru,
    "LSTM-LogExp": pred_lstm_l,
    "GRU-LogExp": pred_gru_l,
    "HAR-Cholesky_ker": pred_har_chol_ker,
    "HAR-LogExp_ker": pred_har_log_ker,
    "DCC-real_ker": pred_dcc_re_ker,
    "LSTM-Cholesky_ker": pred_lstm_ker,
    "GRU-Cholesky_ker": pred_gru_ker,
    "LSTM-LogExp_ker": pred_lstm_l_ker,
    "GRU-LogExp_ker": pred_gru_l_ker,
    "True": realized_rc
}

results = {}
for name, pred in predictions.items():
    results[name] = {
        "RMSE":  compute_rmse(pred, realized_rc),
        "QLIKE": compute_qlike(pred, realized_rc)
    }
pd.DataFrame(results).T.round(6)

,RMSE,QLIKE
HAR-Cholesky,0.000424,4.046009
HAR-LogExp,0.000412,3.785637
DCC-st,0.000593,4.083967
DCC-real,0.000412,4.419462
LSTM-Cholesky,0.000590,8.071443
GRU-Cholesky,0.000512,7.771111
LSTM-LogExp,0.000428,4.772749
GRU-LogExp,0.000425,5.169769
HAR-Cholesky_ker,0.000429,4.736520
HAR-LogExp_ker,0.000415,4.400999


In [8]:
import numpy as np
saved = np.loadtxt("data/pred_lstm_rc_sub.txt")
diag_idx = [0, 2, 5, 9, 14, 20, 27, 35]
diags = saved[:, diag_idx]
print("predicted diag — min:", diags.min(), "max:", diags.max(), "median:", np.median(diags))
print("how many rows have any diag < 1e-8:", (diags < 1e-8).any(axis=1).sum())

predicted diag — min: 1.8012276996159926e-05 max: 0.0003887892817147076 median: 0.00010745739928097464
how many rows have any diag < 1e-8: 0


In [9]:
data = realized_rc

if data.ndim == 1:
    data = data.reshape(1, -1)

n_rows = data.shape[0]

matrices = data.reshape(n_rows, 8, 8)

diagonals = np.diagonal(matrices, axis1=1, axis2=2)

sigma_annual = np.sqrt(diagonals * 252)

print(f"Processed {n_rows} rows.")
print(f"Shape of sigma_annual: {sigma_annual.shape} (Rows x Diagonal Elements)")
print("\nFirst row of annualized sigmas:")
print(sigma_annual[0])

Processed 600 rows.
Shape of sigma_annual: (600, 8) (Rows x Diagonal Elements)

First row of annualized sigmas:
[0.16539442 0.34611459 0.11066219 0.10829407 0.16253942 0.08657319
 0.21919543 0.18506617]


In [10]:
print(f"Mean realized: {realized_rc.mean():.2e}")
for name, pred in predictions.items():
    avg_err = np.mean(np.abs(pred - realized_rc))
    print(f"{name} mean abs error: {avg_err:.2e}, ratio: {avg_err/realized_rc.mean():.2f}x")
    print(f"{name} pred mean: {pred.mean():.2e}")

Mean realized: 5.04e-05
HAR-Cholesky mean abs error: 3.22e-05, ratio: 0.64x
HAR-Cholesky pred mean: 5.36e-05
HAR-LogExp mean abs error: 2.87e-05, ratio: 0.57x
HAR-LogExp pred mean: 4.40e-05
DCC-st mean abs error: 5.07e-05, ratio: 1.01x
DCC-st pred mean: 8.63e-05
DCC-real mean abs error: 3.09e-05, ratio: 0.61x
DCC-real pred mean: 5.25e-05
LSTM-Cholesky mean abs error: 5.51e-05, ratio: 1.09x
LSTM-Cholesky pred mean: 8.45e-05
GRU-Cholesky mean abs error: 4.10e-05, ratio: 0.81x
GRU-Cholesky pred mean: 6.10e-05
LSTM-LogExp mean abs error: 3.15e-05, ratio: 0.62x
LSTM-LogExp pred mean: 4.82e-05
GRU-LogExp mean abs error: 2.99e-05, ratio: 0.59x
GRU-LogExp pred mean: 4.27e-05
HAR-Cholesky_ker mean abs error: 3.24e-05, ratio: 0.64x
HAR-Cholesky_ker pred mean: 5.17e-05
HAR-LogExp_ker mean abs error: 2.84e-05, ratio: 0.56x
HAR-LogExp_ker pred mean: 4.15e-05
DCC-real_ker mean abs error: 3.34e-05, ratio: 0.66x
DCC-real_ker pred mean: 6.04e-05
LSTM-Cholesky_ker mean abs error: 7.18e-05, ratio: 1.42x


In [11]:
import numpy as np

saved = np.loadtxt("data/pred_lstm_rc_sub.txt")  # (T, 36)
n = 8

def vec_to_sym_np(v, n=8):
    M = np.zeros((n, n))
    k = 0
    for i in range(n):
        for j in range(i + 1):
            M[i, j] = v[k]; M[j, i] = v[k]
            k += 1
    return M

mats = np.stack([vec_to_sym_np(saved[t]) for t in range(len(saved))])
eigs = np.linalg.eigvalsh(mats)                       # (T, n)
min_eigs = eigs[:, 0]

print("min eigenvalue across all preds:", min_eigs.min())
print("count with min_eig < 1e-8:", (min_eigs < 1e-8).sum())
print("count with min_eig < 1e-10:", (min_eigs < 1e-10).sum())
print("worst 5 min_eigs:", np.sort(min_eigs)[:5])
print("indices of worst 5:", np.argsort(min_eigs)[:5])

# per-observation QLIKE so you see the distribution:
real = np.loadtxt("data/rc_subsampled.txt")[-len(saved):]
real_mats = np.stack([vec_to_sym_np(real[t]) for t in range(len(real))])
qlikes = []
for t in range(len(saved)):
    try:
        Hinv = np.linalg.inv(mats[t])
        A = real_mats[t] @ Hinv
        s, ld = np.linalg.slogdet(A)
        if s <= 0: continue
        qlikes.append(np.trace(A) - ld - n)
    except np.linalg.LinAlgError:
        continue
qlikes = np.array(qlikes)
print(f"QLIKE — mean: {qlikes.mean():.2f}, median: {np.median(qlikes):.4f}, "
      f"95th: {np.percentile(qlikes, 95):.2f}, max: {qlikes.max():.2e}")

min eigenvalue across all preds: 1.0000000154251612e-05
count with min_eig < 1e-8: 0
count with min_eig < 1e-10: 0
worst 5 min_eigs: [1.00000002e-05 1.00000002e-05 1.00000009e-05 1.00000019e-05
 1.00000030e-05]
indices of worst 5: [504 518 336 442 367]
QLIKE — mean: 5.50, median: 4.4609, 95th: 11.75, max: 6.14e+01
